# Advanced Retrieval Evaluation

This notebook evaluates different retrieval strategies using RAGAS and LangSmith metrics.

## Setup and Configuration

In [ ]:
import getpass
import os
from dataloader import load_data
from uuid import uuid4

use_api_keys_input = True # Set to True to use API keys from input

if use_api_keys_input:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")
    os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")
    os.environ["LANGCHAIN_TRACING_V2"] = "true"
    os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")
    os.environ["LANGCHAIN_PROJECT"] = f"AIM - ADVANCED RETRIEVAL - {uuid4().hex[0:8]}"


## Import Dependencies


In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
from retrievers import get_retrieval_chains_and_wrappers
from rag_prompt import get_rag_prompt
from langchain_qdrant import Qdrant
from langchain_experimental.text_splitter import SemanticChunker
from langchain_core.stores import InMemoryStore
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient, models
from langchain_text_splitters import RecursiveCharacterTextSplitter
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.testset import TestsetGenerator
from ragas import EvaluationDataset
from ragas import evaluate as evaluate_ragas, RunConfig
from ragas.metrics import (
    LLMContextPrecisionWithReference,
    LLMContextRecall,
    ContextEntityRecall,
)
from langsmith.evaluation import LangChainStringEvaluator, evaluate as evaluate_langsmith
from langsmith import Client as ClientLangSmith
import numpy as np


## Configuration

Set the mode for retrieval strategy. Options:
- `"naive-baseline"`: Standard vector similarity search
- `"semantic"`: Semantic chunking with percentile thresholding

In [ ]:
MODE = "naive-baseline"  # semantic


## Load the Data

Load the loan complaint data for evaluation.


In [ ]:
loan_complaint_data = load_data(num_docs=50)


## Create Core Components

Initialize the embeddings model, chat model, and RAG prompt template.

In [ ]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
chat_model = ChatOpenAI(model="gpt-4.1-nano")
rag_prompt = get_rag_prompt()


## Setup Vector Store Based on Mode

Configure the vector store based on the selected mode - either naive baseline or semantic chunking.

In [ ]:
if MODE == "naive-baseline":
    # ===============================
    # Naive Retrieval
    # ===============================

    vectorstore = Qdrant.from_documents(
        loan_complaint_data,
        embeddings,
        location=":memory:",
        collection_name="LoanComplaints"
    )
    
elif MODE == "semantic":
    # ===============================
    # Semantic Retrieval
    # ===============================

    semantic_chunker = SemanticChunker(
        embeddings,
        breakpoint_threshold_type="percentile"
    )
    loan_complaint_data = semantic_chunker.split_documents(loan_complaint_data)
    semantic_vectorstore = Qdrant.from_documents(
        loan_complaint_data,
        embeddings,
        location=":memory:",
        collection_name="Loan_Complaint_Data_Semantic_Chunks"
    )
    vectorstore = semantic_vectorstore
else:
    raise ValueError(f"Invalid mode: {MODE}")


## Create RAGAS Dataset

Generate a golden dataset using RAGAS for evaluation purposes.

In [ ]:
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
golden_dataset = generator.generate_with_langchain_docs(loan_complaint_data, testset_size=50)

# View the dataset
# golden_dataset.to_pandas()


## Setup Parent Document Retrieval

Configure the parent document retrieval strategy with child chunking.

In [ ]:
# Create the retriever - parent document retrieval
parent_docs = loan_complaint_data
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

client_qdrant = QdrantClient(location=":memory:")
client_qdrant.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", 
    embedding=OpenAIEmbeddings(model="text-embedding-3-small"), 
    client=client_qdrant
)

in_memory_store = InMemoryStore()


## Configure Retrievers

Set up the configuration for different retrieval strategies.

In [ ]:
retrievers_config = {
    "naive": {
        "vectorstore": vectorstore
    },
    "parent_document": {
        "vectorstore": parent_document_vectorstore,
        "in_memory_store": in_memory_store,
        "child_splitter": child_splitter
    }
}


## Create RAG Retrievers and Chains

Generate the retrieval chains and wrappers for evaluation.

In [ ]:
chains, wrappers = get_retrieval_chains_and_wrappers(
    retrievers_config, 
    loan_complaint_data, 
    rag_prompt, 
    chat_model, 
    MODE
)


## Initialize Evaluation Results Storage

Prepare dictionaries to store evaluation results.

In [ ]:
eval_full_results = {}
eval_summary_results = {}
eval_langsmith_raw_results = {}
eval_langsmith_summary_results = {}


## Run RAGAS Evaluation

Evaluate each retrieval strategy using RAGAS metrics for retrieval performance.

In [ ]:
for chain_name in chains.keys():
    print(f"\n=== Evaluating {chain_name} with RAGAS ===")
    
    # Prepare dataset for this chain
    golden_dataset_active_copy = golden_dataset.copy()

    # Generate responses for evaluation
    for test_row in golden_dataset_active_copy:
        response = wrappers[chain_name](test_row.eval_sample.user_input)
        test_row.eval_sample.response = response["response"]
        test_row.eval_sample.retrieved_contexts = [context.page_content for context in response["context"]]

    evaluation_active_dataset = EvaluationDataset.from_pandas(golden_dataset_active_copy.to_pandas())
    evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-mini"))

    custom_run_config = RunConfig(timeout=360)

    # Run evaluation using only RETRIEVAL metrics
    print("Running RAGAS evaluation...")
    eval_result_active = evaluate_ragas(
        dataset=evaluation_active_dataset,
        metrics=[LLMContextPrecisionWithReference(), LLMContextRecall(), ContextEntityRecall()],
        llm=evaluator_llm,
        run_config=custom_run_config
    )

    eval_full_results[chain_name] = eval_result_active

    # Convert to DataFrames and compute statistics
    df_eval_result_active = eval_result_active.to_pandas()
    eval_result_active_means = df_eval_result_active.mean(numeric_only=True)
    eval_result_active_stds = df_eval_result_active.std(numeric_only=True)

    eval_summary_results[chain_name] = {
        "means": eval_result_active_means,
        "stds": eval_result_active_stds
    }
    
    print(f"RAGAS Results for {chain_name}:")
    print(eval_result_active_means)


## LangSmith Evaluation

Run additional evaluation using LangSmith for cost and latency analysis.


In [ ]:
for chain_name in chains.keys():
    print(f"\n=== LangSmith Evaluation for {chain_name} ===")
    
    # Setup LangSmith client and dataset
    client_langsmith = ClientLangSmith()
    dataset_name = f"Advanced Retrieval - {chain_name}"

    langsmith_dataset = client_langsmith.create_dataset(
        dataset_name=dataset_name,
        description=f"Advanced Retrieval - {chain_name}"
    )

    # Add examples to LangSmith dataset
    evaluation_active_dataset = EvaluationDataset.from_pandas(golden_dataset.to_pandas())
    for data_row in evaluation_active_dataset.to_pandas().iterrows():
        client_langsmith.create_example(
            inputs={
                "question": data_row[1]["user_input"]
            },
            outputs={
                "answer": data_row[1]["reference"]
            },
            metadata={
                "context": data_row[1]["reference_contexts"]
            },
            dataset_id=langsmith_dataset.id
        )

    # Setup evaluators
    eval_llm_langsmith = ChatOpenAI(model="gpt-4.1-mini")
    qa_evaluator = LangChainStringEvaluator("qa", config={"llm": eval_llm_langsmith})

    context_relevance_evaluator = LangChainStringEvaluator(
        "labeled_criteria",
        config={
            "criteria": {
                "context_relevance": (
                    "How relevant is the retrieved context to the input question?"
                    " Rate it based on whether the context helps answer the question directly,"
                    " contains distracting/unrelated info, or is missing key facts."
                )
            },
            "llm": eval_llm_langsmith,
        },
        prepare_data=lambda run, example: {
            "prediction": run.inputs.get("context", ""),
            "reference": example.outputs.get("answer", ""),
            "input": example.inputs["question"],
        },
    )

    # Run LangSmith evaluation
    print("Running LangSmith evaluation...")
    eval_langsmith_result_active = evaluate_langsmith(
        chains[chain_name].invoke,
        data=evaluation_active_dataset,
        evaluators=[
            qa_evaluator,
            context_relevance_evaluator,
        ],
        metadata={"revision_id": "default_chain_init"},
    )
    
    # Calculate performance metrics
    all_latencies = [res.run.execution_time for res in eval_langsmith_result_active]
    all_costs = [res.run.cost for res in eval_langsmith_result_active]

    average_latency = np.mean(all_latencies)
    total_cost = sum(all_costs)

    eval_langsmith_raw_results[chain_name] = eval_langsmith_result_active
    eval_langsmith_summary_results[chain_name] = {
        "average_latency": average_latency,
        "total_cost": total_cost
    }
    
    print(f"Average Latency: {average_latency:.3f}s")
    print(f"Total Cost: ${total_cost:.4f}")


## Results Summary

Display comprehensive evaluation results for all retrieval strategies.

In [ ]:
print("\n" + "=" * 50)
print("FINAL EVALUATION SUMMARY")
print("=" * 50)

for chain_name in chains.keys():
    print(f"\n--- {chain_name.upper()} ---")
    
    # RAGAS metrics
    print("\nRAGAS Metrics (Mean ± Std):")
    means = eval_summary_results[chain_name]["means"]
    stds = eval_summary_results[chain_name]["stds"]
    
    for metric in means.index:
        print(f"  {metric}: {means[metric]:.3f} ± {stds[metric]:.3f}")
    
    # LangSmith metrics
    print("\nPerformance Metrics:")
    langsmith_results = eval_langsmith_summary_results[chain_name]
    print(f"  Average Latency: {langsmith_results['average_latency']:.3f}s")
    print(f"  Total Cost: ${langsmith_results['total_cost']:.4f}")


In [ ]:
# Create heatmap visualization
def create_heatmap_visualization(eval_summary_results):
    """
    Create a heatmap showing metric performance across chains
    """
    # Prepare data for heatmap
    heatmap_data = []
    chains = list(eval_summary_results.keys())
    metrics = ['LLMContextPrecisionWithReference', 'LLMContextRecall', 'ContextEntityRecall']
    
    for chain in chains:
        means = eval_summary_results[chain]["means"]
        row_data = [means[metric] for metric in metrics]
        heatmap_data.append(row_data)
    
    # Create DataFrame for heatmap
    df_heatmap = pd.DataFrame(
        heatmap_data, 
        index=[chain.replace('_', ' ').title() for chain in chains],
        columns=['Context Precision', 'Context Recall', 'Entity Recall']
    )
    
    # Create the heatmap
    plt.figure(figsize=(10, 6))
    sns.heatmap(df_heatmap, 
                annot=True, 
                cmap='RdYlBu_r',
                fmt='.3f',
                linewidths=0.5,
                cbar_kws={'label': 'Score'},
                square=True)
    
    plt.title('RAG Evaluation Metrics Heatmap\n(Darker = Better Performance)', 
              fontsize=14, fontweight='bold', pad=20)
    plt.xlabel('Metrics', fontsize=12, fontweight='bold')
    plt.ylabel('Retrieval Strategies', fontsize=12, fontweight='bold')
    plt.xticks(rotation=45)
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()
    
    return df_heatmap

# Create performance comparison radar chart
def create_radar_chart(eval_summary_results):
    """
    Create a radar chart comparing all metrics across chains
    """
    import numpy as np
    
    chains = list(eval_summary_results.keys())
    metrics = ['LLMContextPrecisionWithReference', 'LLMContextRecall', 'ContextEntityRecall']
    metric_labels = ['Context\nPrecision', 'Context\nRecall', 'Entity\nRecall']
    
    # Number of metrics
    N = len(metrics)
    
    # Compute angle for each metric
    angles = [n / float(N) * 2 * np.pi for n in range(N)]
    angles += angles[:1]  # Complete the circle
    
    # Create the plot
    fig, ax = plt.subplots(figsize=(10, 8), subplot_kw=dict(projection='polar'))
    
    colors = ['#2E86AB', '#A23B72', '#F18F01', '#E71D36']
    
    for i, chain in enumerate(chains):
        means = eval_summary_results[chain]["means"]
        values = [means[metric] for metric in metrics]
        values += values[:1]  # Complete the circle
        
        ax.plot(angles, values, 'o-', linewidth=2, 
                label=chain.replace('_', ' ').title(), color=colors[i % len(colors)])
        ax.fill(angles, values, alpha=0.25, color=colors[i % len(colors)])
    
    # Customize the radar chart
    ax.set_xticks(angles[:-1])
    ax.set_xticklabels(metric_labels)
    ax.set_ylim(0, 1)
    ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
    ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'])
    ax.grid(True)
    
    plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.0))
    plt.title('RAG Evaluation Metrics - Radar Chart\n(Further from center = Better)', 
              fontsize=14, fontweight='bold', pad=30)
    
    plt.tight_layout()
    plt.show()

# Generate alternative visualizations
print("\nCreating heatmap visualization...")
heatmap_df = create_heatmap_visualization(eval_summary_results)

print("\nCreating radar chart...")
create_radar_chart(eval_summary_results)


In [ ]:
# Performance metrics visualization
def plot_performance_metrics(eval_langsmith_summary_results):
    """
    Plot cost and latency performance metrics
    """
    chains = list(eval_langsmith_summary_results.keys())
    latencies = [eval_langsmith_summary_results[chain]["average_latency"] for chain in chains]
    costs = [eval_langsmith_summary_results[chain]["total_cost"] for chain in chains]
    
    # Create subplot for cost and latency
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # Latency plot
    bars1 = ax1.bar(chains, latencies, color='#2E86AB', alpha=0.8)
    ax1.set_title('Average Latency by Retrieval Strategy', fontsize=14, fontweight='bold')
    ax1.set_xlabel('Retrieval Strategy', fontsize=12)
    ax1.set_ylabel('Latency (seconds)', fontsize=12)
    ax1.set_xticklabels([chain.replace('_', ' ').title() for chain in chains], rotation=45)
    ax1.grid(True, alpha=0.3)
    
    # Add value labels on bars
    for bar, latency in zip(bars1, latencies):
        height = bar.get_height()
        ax1.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                f'{latency:.3f}s', ha='center', va='bottom', fontsize=10)
    
    # Cost plot
    bars2 = ax2.bar(chains, costs, color='#A23B72', alpha=0.8)
    ax2.set_title('Total Cost by Retrieval Strategy', fontsize=14, fontweight='bold')
    ax2.set_xlabel('Retrieval Strategy', fontsize=12)
    ax2.set_ylabel('Total Cost ($)', fontsize=12)
    ax2.set_xticklabels([chain.replace('_', ' ').title() for chain in chains], rotation=45)
    ax2.grid(True, alpha=0.3)
    
    # Add value labels on bars
    for bar, cost in zip(bars2, costs):
        height = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width()/2., height + 0.0001,
                f'${cost:.4f}', ha='center', va='bottom', fontsize=10)
    
    plt.tight_layout()
    plt.show()
    
    # Create performance summary table
    perf_table_data = []
    for chain in chains:
        results = eval_langsmith_summary_results[chain]
        perf_table_data.append({
            'Chain': chain.replace('_', ' ').title(),
            'Avg Latency (s)': f"{results['average_latency']:.3f}",
            'Total Cost ($)': f"{results['total_cost']:.4f}",
            'Cost per Query ($)': f"{results['total_cost']/50:.6f}"  # Assuming 50 queries
        })
    
    perf_df = pd.DataFrame(perf_table_data)
    print("\n" + "="*60)
    print("PERFORMANCE METRICS SUMMARY")
    print("="*60)
    print(perf_df.to_string(index=False))
    
    return perf_df

# Create combined effectiveness vs efficiency plot
def plot_effectiveness_vs_efficiency(eval_summary_results, eval_langsmith_summary_results):
    """
    Create a scatter plot showing effectiveness vs efficiency
    """
    chains = list(eval_summary_results.keys())
    
    # Calculate overall effectiveness score (average of all RAGAS metrics)
    effectiveness_scores = []
    for chain in chains:
        means = eval_summary_results[chain]["means"]
        avg_score = (means['LLMContextPrecisionWithReference'] + 
                    means['LLMContextRecall'] + 
                    means['ContextEntityRecall']) / 3
        effectiveness_scores.append(avg_score)
    
    # Get efficiency metrics (inverse of latency for "efficiency")
    latencies = [eval_langsmith_summary_results[chain]["average_latency"] for chain in chains]
    costs = [eval_langsmith_summary_results[chain]["total_cost"] for chain in chains]
    
    # Create the scatter plot
    plt.figure(figsize=(12, 8))
    
    # Create scatter plot with cost as bubble size
    colors = ['#2E86AB', '#A23B72', '#F18F01', '#E71D36']
    
    for i, chain in enumerate(chains):
        plt.scatter(effectiveness_scores[i], 1/latencies[i], 
                   s=costs[i]*50000, alpha=0.6, 
                   color=colors[i % len(colors)],
                   label=f"{chain.replace('_', ' ').title()}\n(Cost: ${costs[i]:.4f})")
    
    plt.xlabel('Effectiveness Score\n(Average of RAGAS Metrics)', fontsize=12, fontweight='bold')
    plt.ylabel('Efficiency Score\n(1/Average Latency)', fontsize=12, fontweight='bold')
    plt.title('Effectiveness vs Efficiency Analysis\n(Bubble size = Total Cost)', 
              fontsize=14, fontweight='bold')
    plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    
    # Add annotations for each point
    for i, chain in enumerate(chains):
        plt.annotate(f'{effectiveness_scores[i]:.3f}', 
                    (effectiveness_scores[i], 1/latencies[i]),
                    xytext=(5, 5), textcoords='offset points', fontsize=9)
    
    plt.tight_layout()
    plt.show()

# Generate performance visualizations
if 'eval_langsmith_summary_results' in globals() and eval_langsmith_summary_results:
    print("Creating performance metrics visualization...")
    perf_df = plot_performance_metrics(eval_langsmith_summary_results)
    
    print("\nCreating effectiveness vs efficiency analysis...")
    plot_effectiveness_vs_efficiency(eval_summary_results, eval_langsmith_summary_results)
else:
    print("LangSmith results not available yet. Run the LangSmith evaluation section first.")
